# Modeling Regional Segments

Segment models are supporting evidence. The emphasis is heterogeneity rather than unnecessary model complexity.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import CLEAN_SEGMENTS, FIGURES, TABLES, TARGET_COLUMNS, SEGMENT_COLUMNS, SERIES_COLORS, SERIES_LABELS
from src.plotting import annotate_events, save_figure, set_academic_style

set_academic_style()
df = pd.read_csv(CLEAN_SEGMENTS, parse_dates=["date"]).set_index("date").sort_index()
df.index.freq = "MS"

from src.models.evaluation import run_all

## Segment Forecast Evaluation

Observation: regions differ in recovery level and volatility. Statistical implication: a model that performs well for total arrivals may not dominate every segment. Tourism implication: source-market planning should use segment-specific forecast risk.

In [ ]:
metrics, best, forecasts = run_all(df, targets=SEGMENT_COLUMNS)
metrics.to_csv(TABLES / "model_metrics_segments.csv", index=False)
best.to_csv(TABLES / "best_models_segments.csv", index=False)
best

In [ ]:
fig, axes = plt.subplots(len(SEGMENT_COLUMNS), 1, figsize=(9, 11), sharex=True)
for ax, col in zip(axes, SEGMENT_COLUMNS):
    ax.plot(df.loc["2019":].index, df.loc["2019":, col], color=SERIES_COLORS[col], label="Observed")
    row = best[best["target"].eq(col)].iloc[0]
    sub = forecasts[(forecasts["target"].eq(col)) & (forecasts["model"].eq(row["model"]))].copy()
    sub["date"] = pd.to_datetime(sub["date"])
    ax.plot(sub["date"], sub["forecast"], color="#2f3437", alpha=0.85, label=row["model"])
    ax.set_title(f"{SERIES_LABELS[col]}: Best Out-of-Sample Forecast")
    ax.set_ylabel("Arrivals")
axes[0].legend(fontsize=8)
save_figure(fig, FIGURES / "14_segment_forecasts.png")
plt.show()

## Interpretation

Observation: Asia dominates the total path, while smaller regions show sharper proportional swings. Statistical implication: segment residual variance is heterogeneous. Tourism implication: market-specific policies are necessary for resilient recovery planning.